# APHC


# Direct/FIML Weibull ALT benchmark — fixed 2 censoring settings per parameter setting

This notebook benchmarks the original **two-step** estimator against a **direct joint MLE/FIML** estimator of

`[a0, a1, a2, c0, c1, c2]`

with

- `lambda(T,V) = a0 + a1/T + a2*log(V)`
- `alpha(T,V) = c0 + c1/T + c2*log(V)`

using **Nelder–Mead** and **Powell**.

The notebook does **not** resimulate observations. It reads the exact saved `Simulated_data/pair_*.npz` files from the original simulation so the benchmark is paired and directly comparable.

## Fixed 2 censoring settings per parameter setting

| Dataset | n | m | Planned observed | Purpose |
|---|---:|---:|---:|---|
| `m60_n240_balanced` | 240 | 60 | 25% | Heavy-censoring benchmark |
| `m180_n240_balanced` | 240 | 180 | 75% | Light-censoring benchmark |


The benchmark uses the same two fixed-n censoring settings for each of the four parameter settings: 25% observed and 75% observed, both with n=240. This gives 4 × 2 = 8 total benchmark scenarios and a clean contrast between heavy and light censoring.

## Run strategy

The default repetition batch is `0:100`. On an 8 GB laptop run:

- 0–100
- 100–200
- ...
- 900–1000

The run is resumable. Existing `(Repetition, Method)` results are skipped.

The direct benchmark is **fully independent of the two-stage estimator**. Neither two-stage regression estimates nor pair-level first-stage MLEs nor true simulation coefficients are used for initialization. Both Nelder–Mead and Powell receive the same four fixed, pre-specified starting vectors. Each optimizer is run from every start, and its converged solution with the smallest negative log-likelihood is retained.


## Numerical safeguards in this reviewed version

- No two-stage estimate, first-stage MLE, true coefficient, or current-repetition estimate is used to initialize either optimizer.
- Both methods receive the same four fixed starting vectors for every repetition.
- Multi-start optimization is used: all four starts are tried independently, and the best converged likelihood is retained for each optimizer.
- Optimization is performed in an exactly equivalent standardized-predictor coefficient parameterization to reduce ill-conditioning; estimates are transformed back to `[a0,a1,a2,c0,c1,c2]`.
- PHC/APHC saved-data structure is validated before optimization.
- The likelihood rejects any candidate producing nonpositive shape or scale at any of the 50 stress pairs.
- `LogLik_kernel` omits only parameter-independent censoring combinatorial constants, so it is valid for MLE and within-scenario optimizer comparison, but should not be compared across different censoring designs.
- `Optimizer_agreement.csv` compares Nelder-Mead and Powell when both converge.


In [7]:

from pathlib import Path
import math
import time
import gc
import warnings

import numpy as np
import pandas as pd
from scipy.optimize import minimize

CENSORING = "APHC"
SOURCE_ROOT = Path("Results") / CENSORING
OUTPUT_ROOT = Path("Results") / "FIML_Benchmark_2perSetting_INDEPENDENT" / CENSORING
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

FIXED_DATASETS = ["m60_n240_balanced", "m180_n240_balanced"]

# Use all parameter settings found in the original results
# (normally baseline, weaker_temperature, positive_voltage, weaker_stress).
SELECTED_PARAMETER_SETTINGS = None

# Run in repetition batches on a laptop.
START_REP = 0
END_REP = 100

METHODS = ("Nelder-Mead", "Powell")

# ============================================================
# INDEPENDENT MULTI-START INITIALIZATION
# ============================================================
# These starts are fixed in advance and are NOT obtained from:
# - the two-stage regression,
# - pair-level first-stage MLEs,
# - true simulation coefficients,
# - or the current Monte Carlo data.
#
# Both Nelder-Mead and Powell receive exactly the same set.
# For each method/repetition, all starts are optimized independently
# and the converged solution with the smallest negative log-likelihood
# is retained.
INDEPENDENT_STARTS = [
    ("S1_constant_1",
     np.array([1.0, 0.0, 0.0, 1.0, 0.0, 0.0])),
    ("S2_constant_5",
     np.array([2.0, 0.0, 0.0, 2.0, 0.0, 0.0])),
    # ("S1_constant_1",
    #  np.array([1.0, 0.0, 0.0, 1.0, 0.0, 0.0])),
    # ("S2_constant_5",
    #  np.array([5.0, 0.0, 0.0, 5.0, 0.0, 0.0])),
    # ("S3_mild_negative_voltage",
    #  np.array([5.0, 50.0, -1.0, 5.0, 50.0, -1.0])),
    # ("S4_mild_positive_voltage",
    #  np.array([2.0, 25.0, 0.25, 2.0, 25.0, 0.25])),
]

CHECKPOINT_EVERY = 5

# Numerical Hessian CIs are implemented but are expensive.
# Leave False for the optimizer/RMSE benchmark; turn on only when needed.
COMPUTE_CI = False

WRITE_EXCEL = True

COEFFICIENTS = ["a0", "a1", "a2", "c0", "c1", "c2"]
PENALTY = 1e100
Z95 = 1.959963984540054
Z99 = 2.5758293035489004

NELDER_MEAD_OPTIONS = {
    "maxiter": 2500,
    "xatol": 1e-7,
    "fatol": 1e-7,
    "adaptive": True,
}

POWELL_OPTIONS = {
    "maxiter": 1500,
    "xtol": 1e-6,
    "ftol": 1e-7,
}

print("Source:", SOURCE_ROOT.resolve())
print("Output:", OUTPUT_ROOT.resolve())
print("Fixed datasets:", FIXED_DATASETS)


Source: C:\Users\NetComp\Documents\RJ\Results\APHC
Output: C:\Users\NetComp\Documents\RJ\Results\FIML_Benchmark_2perSetting_INDEPENDENT\APHC
Fixed datasets: ['m60_n240_balanced', 'm180_n240_balanced']


In [8]:

# ============================================================
# Validate source folders and fixed selection
# ============================================================

settings_files = sorted(SOURCE_ROOT.glob("*/*/Settings.csv"))

if not settings_files:
    raise FileNotFoundError(
        f"No Settings.csv files found under {SOURCE_ROOT.resolve()}"
    )

available_parameter_settings = sorted({
    p.parent.parent.name for p in settings_files
})
available_datasets = sorted({
    p.parent.name for p in settings_files
})

missing_datasets = [
    name for name in FIXED_DATASETS
    if name not in available_datasets
]

if missing_datasets:
    raise FileNotFoundError(
        "These fixed benchmark datasets are missing from "
        f"{SOURCE_ROOT}: {missing_datasets}"
    )

if SELECTED_PARAMETER_SETTINGS is None:
    selected_parameter_settings = available_parameter_settings
else:
    selected_parameter_settings = list(SELECTED_PARAMETER_SETTINGS)
    missing_parameters = sorted(
        set(selected_parameter_settings)
        - set(available_parameter_settings)
    )
    if missing_parameters:
        raise ValueError(
            f"Unknown parameter settings: {missing_parameters}"
        )

# Preserve the explicitly chosen dataset order.
scenario_paths = []
for parameter_name in selected_parameter_settings:
    for dataset_name in FIXED_DATASETS:
        p = (
            SOURCE_ROOT
            / parameter_name
            / dataset_name
            / "Settings.csv"
        )
        if not p.exists():
            raise FileNotFoundError(p)
        scenario_paths.append(p)

selection_rows = []
for dataset_name in FIXED_DATASETS:
    p = (
        SOURCE_ROOT
        / selected_parameter_settings[0]
        / dataset_name
        / "Settings.csv"
    )
    s = pd.read_csv(p).iloc[0]
    selection_rows.append({
        "Dataset": dataset_name,
        "n": int(s["n"]),
        "m": int(s["m"]),
        "Planned_observed_fraction": (
            float(s["m"]) / float(s["n"])
        ),
    })

selection_df = pd.DataFrame(selection_rows)
selection_df.to_csv(
    OUTPUT_ROOT / "Selected_2_datasets.csv",
    index=False,
)

display(selection_df)
print("Parameter settings:", selected_parameter_settings)
print("Total benchmark scenarios:", len(scenario_paths))


,Dataset,n,m,Planned_observed_fraction
0,m60_n240_balanced,240,60,0.25
1,m180_n240_balanced,240,180,0.75


Parameter settings: ['baseline', 'positive_voltage', 'weaker_stress', 'weaker_temperature']
Total benchmark scenarios: 8


In [9]:

# ============================================================
# Coefficient model, numerically conditioned optimizer,
# starting values, summaries and diagnostic checks
# ============================================================

def theta_to_stress_parameters(theta, temperatures, voltages):
    """
    Original coefficient parameterization:
      lambda = a0 + a1/T + a2*log(V)
      alpha  = c0 + c1/T + c2*log(V)
    theta = [a0, a1, a2, c0, c1, c2]
    """
    theta = np.asarray(theta, dtype=float)
    a0, a1, a2, c0, c1, c2 = theta

    inv_t = 1.0 / np.asarray(temperatures, dtype=float)
    log_v = np.log(np.asarray(voltages, dtype=float))

    lam = a0 + a1 * inv_t + a2 * log_v
    alpha = c0 + c1 * inv_t + c2 * log_v

    return alpha, lam


def valid_theta(theta, temperatures, voltages):
    theta = np.asarray(theta, dtype=float)

    if theta.shape != (6,) or not np.isfinite(theta).all():
        return False

    alpha, lam = theta_to_stress_parameters(
        theta, temperatures, voltages
    )

    return bool(
        np.isfinite(alpha).all()
        and np.isfinite(lam).all()
        and np.all(alpha > 0)
        and np.all(lam > 0)
    )


def predictor_standardization(temperatures, voltages):
    """
    Standardize the two stress predictors to improve numerical conditioning.
    This is an exact reparameterization; it does not change the likelihood
    or the MLE in the original coefficient scale.
    """
    inv_t = 1.0 / np.asarray(temperatures, dtype=float)
    log_v = np.log(np.asarray(voltages, dtype=float))

    mu1 = float(inv_t.mean())
    mu2 = float(log_v.mean())
    sd1 = float(inv_t.std(ddof=0))
    sd2 = float(log_v.std(ddof=0))

    if (
        not np.isfinite([mu1, mu2, sd1, sd2]).all()
        or sd1 <= 0
        or sd2 <= 0
    ):
        raise ValueError("Invalid stress-predictor standardization.")

    return {
        "mu1": mu1,
        "mu2": mu2,
        "sd1": sd1,
        "sd2": sd2,
    }


def theta_to_eta(theta, stats):
    """
    Convert original coefficients theta to coefficients eta for:
      lambda = b0 + b1*z1 + b2*z2
      alpha  = d0 + d1*z1 + d2*z2
    where z1,z2 are standardized stress predictors.
    """
    a0, a1, a2, c0, c1, c2 = np.asarray(theta, dtype=float)
    mu1, mu2 = stats["mu1"], stats["mu2"]
    sd1, sd2 = stats["sd1"], stats["sd2"]

    return np.array([
        a0 + a1 * mu1 + a2 * mu2,
        a1 * sd1,
        a2 * sd2,
        c0 + c1 * mu1 + c2 * mu2,
        c1 * sd1,
        c2 * sd2,
    ], dtype=float)


def eta_to_theta(eta, stats):
    """Inverse of theta_to_eta."""
    b0, b1, b2, d0, d1, d2 = np.asarray(eta, dtype=float)
    mu1, mu2 = stats["mu1"], stats["mu2"]
    sd1, sd2 = stats["sd1"], stats["sd2"]

    a1 = b1 / sd1
    a2 = b2 / sd2
    a0 = b0 - a1 * mu1 - a2 * mu2

    c1 = d1 / sd1
    c2 = d2 / sd2
    c0 = d0 - c1 * mu1 - c2 * mu2

    return np.array([a0, a1, a2, c0, c1, c2], dtype=float)


def eta_to_theta_jacobian(stats):
    """
    Constant Jacobian d(theta)/d(eta), used to transform
    Hessian-based covariance back to the original coefficients.
    """
    mu1, mu2 = stats["mu1"], stats["mu2"]
    sd1, sd2 = stats["sd1"], stats["sd2"]

    block = np.array([
        [1.0, -mu1 / sd1, -mu2 / sd2],
        [0.0, 1.0 / sd1, 0.0],
        [0.0, 0.0, 1.0 / sd2],
    ])

    J = np.zeros((6, 6), dtype=float)
    J[:3, :3] = block
    J[3:, 3:] = block
    return J


def to_bool_array(series):
    """Robustly parse booleans read either as bool or text."""
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).to_numpy(dtype=bool)

    s = series.astype(str).str.strip().str.lower()
    valid = s.isin(["true", "false"])

    if not valid.all():
        bad = sorted(s.loc[~valid].unique().tolist())
        raise ValueError(f"Unexpected boolean values: {bad}")

    return s.eq("true").to_numpy(dtype=bool)



def independent_start_vectors(temperatures, voltages):
    """
    Return the fixed, pre-specified starting vectors.

    They are identical for PHC/APHC, every parameter setting,
    every censoring setting, every Monte Carlo repetition, and
    both optimization methods.

    The only use of the stress values here is a safety check that
    every pre-specified start gives positive alpha and lambda at all
    50 stress pairs. No estimate or true coefficient is used.
    """
    starts = []

    for start_id, theta in INDEPENDENT_STARTS:
        theta = np.asarray(theta, dtype=float).copy()

        if theta.shape != (6,):
            raise ValueError(
                f"{start_id}: expected six starting coefficients."
            )

        if not valid_theta(theta, temperatures, voltages):
            raise ValueError(
                f"{start_id}: pre-specified start gives a nonpositive "
                "alpha or lambda at one or more stress pairs."
            )

        starts.append((str(start_id), theta))

    if len(starts) < 2:
        raise ValueError(
            "Independent multi-start benchmark requires at least two starts."
        )

    return starts


def optimizer_options(method, eta0):
    if method == "Nelder-Mead":
        options = dict(NELDER_MEAD_OPTIONS)

        # SciPy's default simplex can be too tiny when a starting
        # coefficient is exactly zero. Use an explicit, well-scaled simplex.
        simplex = np.tile(
            np.asarray(eta0, dtype=float),
            (len(eta0) + 1, 1),
        )
        for k in range(len(eta0)):
            step = max(0.05 * abs(eta0[k]), 0.05)
            simplex[k + 1, k] += step

        options["initial_simplex"] = simplex
        return options

    if method == "Powell":
        return dict(POWELL_OPTIONS)

    raise ValueError(f"Unknown optimizer: {method}")


def fit_direct_mle(
    objective_theta,
    theta0,
    temperatures,
    voltages,
    method,
):
    """
    Direct MLE using an exactly equivalent standardized-predictor
    coefficient parameterization for numerical stability.
    """
    theta0 = np.asarray(theta0, dtype=float)

    if not valid_theta(
        theta0, temperatures, voltages
    ):
        raise ValueError("Invalid starting coefficient vector.")

    stats = predictor_standardization(
        temperatures, voltages
    )
    eta0 = theta_to_eta(theta0, stats)

    def objective_eta(eta):
        theta = eta_to_theta(eta, stats)
        return objective_theta(theta)

    objective_start = float(objective_eta(eta0))

    if (
        not np.isfinite(objective_start)
        or objective_start >= PENALTY / 10
    ):
        raise ValueError("Invalid likelihood at starting values.")

    start_time = time.perf_counter()

    result = minimize(
        objective_eta,
        eta0,
        method=method,
        options=optimizer_options(method, eta0),
    )

    runtime = time.perf_counter() - start_time

    eta_hat = np.asarray(result.x, dtype=float)
    theta_hat = eta_to_theta(eta_hat, stats)
    objective_final = float(objective_theta(theta_hat))

    alpha_hat, lambda_hat = theta_to_stress_parameters(
        theta_hat, temperatures, voltages
    )

    objective_not_worse = bool(
        np.isfinite(objective_final)
        and objective_final
        <= objective_start + 1e-7 * (1.0 + abs(objective_start))
    )

    fit_success = bool(
        result.success
        and valid_theta(
            theta_hat, temperatures, voltages
        )
        and np.isfinite(objective_final)
        and objective_final < PENALTY / 10
        and objective_not_worse
    )

    return {
        "theta_hat": theta_hat,
        "eta_hat": eta_hat,
        "standardization": stats,
        "Fit_success": fit_success,
        "Optimizer_success": bool(result.success),
        "Objective_start": objective_start,
        "Objective": objective_final,
        "Objective_improvement": objective_start - objective_final,
        # This is the log-likelihood kernel: coefficient-independent
        # progressive-censoring combinatorial constants are omitted.
        "LogLik_kernel": (
            -objective_final
            if np.isfinite(objective_final)
            and objective_final < PENALTY / 10
            else np.nan
        ),
        "Min_alpha_hat": (
            float(np.min(alpha_hat))
            if np.isfinite(alpha_hat).all()
            else np.nan
        ),
        "Min_lambda_hat": (
            float(np.min(lambda_hat))
            if np.isfinite(lambda_hat).all()
            else np.nan
        ),
        "NFEV": getattr(result, "nfev", np.nan),
        "NIT": getattr(result, "nit", np.nan),
        "Runtime_sec": runtime,
        "Message": str(result.message),
    }



def fit_direct_mle_multistart(
    objective_theta,
    starts,
    temperatures,
    voltages,
    method,
):
    """
    Run one optimizer independently from every fixed starting vector.

    Selection rule:
      1. Prefer fits satisfying the full Fit_success criteria.
      2. Among those, retain the smallest negative log-likelihood.
      3. If none satisfies Fit_success, retain the finite valid candidate
         with the smallest objective, but keep Fit_success=False.

    Runtime and function-evaluation counts reported for the estimator are
    TOTALS across all attempted starts, because multi-start is part of the
    computational procedure.
    """
    attempts = []

    for start_id, theta0 in starts:
        theta0 = np.asarray(theta0, dtype=float)

        try:
            fit = fit_direct_mle(
                objective_theta,
                theta0,
                temperatures,
                voltages,
                method,
            )

            attempts.append({
                "start_id": start_id,
                "theta0": theta0.copy(),
                "fit": fit,
                "error": "",
            })

        except Exception as error:
            attempts.append({
                "start_id": start_id,
                "theta0": theta0.copy(),
                "fit": None,
                "error": str(error),
            })

    successful = [
        a for a in attempts
        if a["fit"] is not None
        and bool(a["fit"]["Fit_success"])
        and np.isfinite(a["fit"]["Objective"])
    ]

    if successful:
        selected = min(
            successful,
            key=lambda a: float(a["fit"]["Objective"]),
        )
    else:
        finite_candidates = [
            a for a in attempts
            if a["fit"] is not None
            and np.isfinite(a["fit"]["Objective"])
            and a["fit"]["Objective"] < PENALTY / 10
            and valid_theta(
                a["fit"]["theta_hat"],
                temperatures,
                voltages,
            )
        ]

        if not finite_candidates:
            messages = "; ".join(
                f'{a["start_id"]}: {a["error"] or "no valid fit"}'
                for a in attempts
            )
            raise RuntimeError(
                f"All independent starts failed for {method}. {messages}"
            )

        selected = min(
            finite_candidates,
            key=lambda a: float(a["fit"]["Objective"]),
        )

    chosen = dict(selected["fit"])
    selected_theta0 = selected["theta0"].copy()

    total_runtime = float(np.nansum([
        a["fit"]["Runtime_sec"]
        if a["fit"] is not None else np.nan
        for a in attempts
    ]))

    total_nfev = float(np.nansum([
        a["fit"]["NFEV"]
        if a["fit"] is not None else np.nan
        for a in attempts
    ]))

    total_nit = float(np.nansum([
        a["fit"]["NIT"]
        if a["fit"] is not None else np.nan
        for a in attempts
    ]))

    chosen["Selected_start_id"] = selected["start_id"]
    chosen["selected_start_theta"] = selected_theta0
    chosen["Start_source"] = "independent_fixed_multistart"
    chosen["N_starts_attempted"] = len(attempts)
    chosen["N_starts_successful"] = len(successful)

    # Keep the selected run's computational statistics as diagnostics.
    chosen["Selected_start_runtime_sec"] = chosen["Runtime_sec"]
    chosen["Selected_start_NFEV"] = chosen["NFEV"]
    chosen["Selected_start_NIT"] = chosen["NIT"]

    # Estimator cost = cost of all independent starts.
    chosen["Runtime_sec"] = total_runtime
    chosen["NFEV"] = total_nfev
    chosen["NIT"] = total_nit

    return chosen


def _checked_objective(func, x):
    value = float(func(x))
    if (
        not np.isfinite(value)
        or value >= PENALTY / 10
    ):
        raise ValueError(
            "Hessian evaluation left the valid likelihood region."
        )
    return value


def numerical_hessian(func, x, rel_step=1e-4):
    """
    Central finite-difference Hessian.
    Invalid/penalty evaluations are rejected rather than being
    treated as legitimate likelihood values.
    """
    x = np.asarray(x, dtype=float)
    k = len(x)
    h = rel_step * np.maximum(np.abs(x), 1.0)

    H = np.zeros((k, k), dtype=float)
    f0 = _checked_objective(func, x)

    for i in range(k):
        ei = np.zeros(k)
        ei[i] = h[i]

        fp = _checked_objective(func, x + ei)
        fm = _checked_objective(func, x - ei)

        H[i, i] = (
            fp - 2.0 * f0 + fm
        ) / (h[i] ** 2)

        for j in range(i + 1, k):
            ej = np.zeros(k)
            ej[j] = h[j]

            fpp = _checked_objective(func, x + ei + ej)
            fpm = _checked_objective(func, x + ei - ej)
            fmp = _checked_objective(func, x - ei + ej)
            fmm = _checked_objective(func, x - ei - ej)

            value = (
                fpp - fpm - fmp + fmm
            ) / (4.0 * h[i] * h[j])

            H[i, j] = value
            H[j, i] = value

    return 0.5 * (H + H.T)


def ci_from_hessian(
    objective_theta,
    theta_hat,
    temperatures,
    voltages,
):
    """
    Approximate normal CIs from the observed Hessian.

    The Hessian is computed in the standardized eta parameterization
    for conditioning, then covariance is transformed back to the
    original [a0,a1,a2,c0,c1,c2] scale.
    """
    stats = predictor_standardization(
        temperatures, voltages
    )
    eta_hat = theta_to_eta(theta_hat, stats)

    def objective_eta(eta):
        return objective_theta(
            eta_to_theta(eta, stats)
        )

    H_eta = numerical_hessian(
        objective_eta, eta_hat
    )

    if not np.isfinite(H_eta).all():
        raise ValueError("Nonfinite Hessian.")

    np.linalg.cholesky(H_eta)
    covariance_eta = np.linalg.inv(H_eta)

    J = eta_to_theta_jacobian(stats)
    covariance_theta = (
        J @ covariance_eta @ J.T
    )

    diag = np.diag(covariance_theta)

    if (
        np.any(~np.isfinite(diag))
        or np.any(diag <= 0)
    ):
        raise ValueError(
            "Invalid transformed Hessian covariance."
        )

    se = np.sqrt(diag)
    out = {"CI_success": True}

    for k, name in enumerate(COEFFICIENTS):
        out[f"{name}_SE"] = se[k]

        for level, z in (
            (95, Z95),
            (99, Z99),
        ):
            out[f"{name}_CI{level}_lower"] = (
                theta_hat[k] - z * se[k]
            )
            out[f"{name}_CI{level}_upper"] = (
                theta_hat[k] + z * se[k]
            )

    return out


def summarize_estimates(estimates_df, true_theta):
    truth = dict(zip(COEFFICIENTS, true_theta))
    rows = []

    for method in METHODS:
        method_df = estimates_df.loc[
            estimates_df["Method"] == method
        ].copy()

        if method_df.empty:
            continue

        fit_ok = to_bool_array(
            method_df["Fit_success"]
        )

        optimizer_ok = to_bool_array(
            method_df["Optimizer_success"]
        )

        for name in COEFFICIENTS:
            values = method_df[
                f"{name}_est"
            ].to_numpy(float)

            valid = fit_ok & np.isfinite(values)
            valid_values = values[valid]
            errors = valid_values - truth[name]

            row = {
                "Method": method,
                "Coefficient": name,
                "True_value": truth[name],
                "N_repetitions": len(method_df),
                "N_valid_fits": int(valid.sum()),
                "Fit_failure_rate": 1 - valid.mean(),
                "Optimizer_success_rate": optimizer_ok.mean(),
                "Mean_estimate": (
                    valid_values.mean()
                    if valid_values.size else np.nan
                ),
                "Bias": (
                    errors.mean()
                    if errors.size else np.nan
                ),
                "SD": (
                    valid_values.std(ddof=0)
                    if errors.size else np.nan
                ),
                "RMSE": (
                    np.sqrt(np.mean(errors**2))
                    if errors.size else np.nan
                ),
                "Mean_runtime_sec": (
                    method_df["Runtime_sec"].mean()
                ),
                "Mean_NFEV": (
                    method_df["NFEV"].mean()
                ),
                "Mean_objective_improvement": (
                    method_df["Objective_improvement"].mean()
                    if "Objective_improvement" in method_df
                    else np.nan
                ),
            }

            if "CI_success" in method_df.columns:
                ci_ok = to_bool_array(
                    method_df["CI_success"]
                )
            else:
                ci_ok = np.zeros(
                    len(method_df), dtype=bool
                )

            for level in (95, 99):
                lower_col = f"{name}_CI{level}_lower"
                upper_col = f"{name}_CI{level}_upper"

                if (
                    lower_col in method_df.columns
                    and upper_col in method_df.columns
                    and ci_ok.any()
                ):
                    lower = method_df[
                        lower_col
                    ].to_numpy(float)
                    upper = method_df[
                        upper_col
                    ].to_numpy(float)

                    valid_ci = (
                        ci_ok
                        & np.isfinite(lower)
                        & np.isfinite(upper)
                        & (lower <= upper)
                    )

                    covered = (
                        valid_ci
                        & (lower <= truth[name])
                        & (truth[name] <= upper)
                    )

                    row[f"N_valid_CI{level}"] = int(
                        valid_ci.sum()
                    )
                    row[f"Coverage{level}_valid"] = (
                        covered[valid_ci].mean()
                        if valid_ci.any()
                        else np.nan
                    )
                    row[f"Mean_width{level}"] = (
                        (
                            upper[valid_ci]
                            - lower[valid_ci]
                        ).mean()
                        if valid_ci.any()
                        else np.nan
                    )
                else:
                    row[f"N_valid_CI{level}"] = 0
                    row[f"Coverage{level}_valid"] = np.nan
                    row[f"Mean_width{level}"] = np.nan

            rows.append(row)

    return pd.DataFrame(rows)


def optimizer_agreement(
    estimates_df,
    temperatures,
    voltages,
):
    """
    Diagnostic only: when both methods succeed for a repetition,
    compare objective values, coefficients and stress-level predictions.
    """
    rows = []

    for repetition, g in estimates_df.groupby(
        "Repetition", sort=True
    ):
        by_method = {
            str(row["Method"]): row
            for _, row in g.iterrows()
        }

        if not all(m in by_method for m in METHODS):
            continue

        r1 = by_method[METHODS[0]]
        r2 = by_method[METHODS[1]]

        s1 = str(r1["Fit_success"]).lower() == "true"
        s2 = str(r2["Fit_success"]).lower() == "true"

        if not (s1 and s2):
            continue

        th1 = np.array(
            [float(r1[f"{n}_est"]) for n in COEFFICIENTS]
        )
        th2 = np.array(
            [float(r2[f"{n}_est"]) for n in COEFFICIENTS]
        )

        a1, l1 = theta_to_stress_parameters(
            th1, temperatures, voltages
        )
        a2, l2 = theta_to_stress_parameters(
            th2, temperatures, voltages
        )

        scale = np.maximum(
            1.0,
            np.maximum(np.abs(th1), np.abs(th2)),
        )

        rows.append({
            "Repetition": int(repetition),
            "Objective_abs_diff": abs(
                float(r1["Objective"])
                - float(r2["Objective"])
            ),
            "Max_scaled_coefficient_diff": float(
                np.max(np.abs(th1 - th2) / scale)
            ),
            "Max_alpha_prediction_diff": float(
                np.max(np.abs(a1 - a2))
            ),
            "Max_lambda_prediction_diff": float(
                np.max(np.abs(l1 - l2))
            ),
        })

    return pd.DataFrame(rows)


def save_scenario_outputs(
    estimates_df,
    summary_df,
    settings_out,
    output_dir,
    source_scenario_dir,
    temperatures,
    voltages,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    estimates_df.to_csv(
        output_dir / "Direct_MLE_by_repetition.csv",
        index=False,
    )
    summary_df.to_csv(
        output_dir / "Coefficient_summary.csv",
        index=False,
    )

    agreement_df = optimizer_agreement(
        estimates_df,
        temperatures,
        voltages,
    )
    agreement_df.to_csv(
        output_dir / "Optimizer_agreement.csv",
        index=False,
    )

    two_stage_summary = (
        source_scenario_dir
        / "Regression"
        / "Coefficient_summary.csv"
    )

    comparison = None

    if two_stage_summary.exists():
        old = pd.read_csv(two_stage_summary)

        cols = [
            c for c in [
                "Coefficient",
                "True_value",
                "N_valid_fits",
                "Fit_failure_rate",
                "Mean_estimate",
                "Bias",
                "SD",
                "RMSE",
                "Coverage95_valid",
                "Mean_width95",
                "Coverage99_valid",
                "Mean_width99",
            ]
            if c in old.columns
        ]

        if "Coefficient" in cols:
            old = old[cols].copy()

            old = old.rename(columns={
                c: f"TwoStage_{c}"
                for c in old.columns
                if c != "Coefficient"
            })

            comparison = summary_df.merge(
                old,
                on="Coefficient",
                how="left",
            )

            comparison.to_csv(
                output_dir / "Benchmark_vs_two_stage.csv",
                index=False,
            )

    if WRITE_EXCEL:
        with pd.ExcelWriter(
            output_dir / "FIML_Benchmark.xlsx",
            engine="openpyxl",
        ) as writer:
            estimates_df.to_excel(
                writer,
                sheet_name="Estimates",
                index=False,
            )
            summary_df.to_excel(
                writer,
                sheet_name="Summary",
                index=False,
            )
            agreement_df.to_excel(
                writer,
                sheet_name="OptimizerAgreement",
                index=False,
            )
            pd.DataFrame([settings_out]).to_excel(
                writer,
                sheet_name="Settings",
                index=False,
            )
            if comparison is not None:
                comparison.to_excel(
                    writer,
                    sheet_name="Vs_TwoStage",
                    index=False,
                )


def rebuild_global_outputs():
    estimates = []
    summaries = []
    comparisons = []
    agreements = []

    for parameter_name in selected_parameter_settings:
        for dataset_name in FIXED_DATASETS:
            d = (
                OUTPUT_ROOT
                / parameter_name
                / dataset_name
            )

            p = d / "Direct_MLE_by_repetition.csv"
            if p.exists():
                estimates.append(pd.read_csv(p))

            p = d / "Coefficient_summary.csv"
            if p.exists():
                temp = pd.read_csv(p)
                temp["Parameter_setting"] = parameter_name
                temp["Dataset"] = dataset_name
                summaries.append(temp)

            p = d / "Benchmark_vs_two_stage.csv"
            if p.exists():
                temp = pd.read_csv(p)
                temp["Parameter_setting"] = parameter_name
                temp["Dataset"] = dataset_name
                comparisons.append(temp)

            p = d / "Optimizer_agreement.csv"
            if p.exists():
                temp = pd.read_csv(p)
                temp["Parameter_setting"] = parameter_name
                temp["Dataset"] = dataset_name
                agreements.append(temp)

    all_estimates = (
        pd.concat(estimates, ignore_index=True)
        if estimates else pd.DataFrame()
    )

    all_summaries = (
        pd.concat(summaries, ignore_index=True)
        if summaries else pd.DataFrame()
    )

    all_comparisons = (
        pd.concat(comparisons, ignore_index=True)
        if comparisons else pd.DataFrame()
    )

    all_agreements = (
        pd.concat(agreements, ignore_index=True)
        if agreements else pd.DataFrame()
    )

    if not all_estimates.empty:
        all_estimates.to_csv(
            OUTPUT_ROOT / "All_direct_MLE_estimates.csv",
            index=False,
        )

    if not all_summaries.empty:
        all_summaries.to_csv(
            OUTPUT_ROOT / "All_coefficient_summaries.csv",
            index=False,
        )

    if not all_comparisons.empty:
        all_comparisons.to_csv(
            OUTPUT_ROOT / "All_benchmark_vs_two_stage.csv",
            index=False,
        )

    if not all_agreements.empty:
        all_agreements.to_csv(
            OUTPUT_ROOT / "All_optimizer_agreement.csv",
            index=False,
        )

    if WRITE_EXCEL and (
        not all_estimates.empty
        or not all_summaries.empty
    ):
        with pd.ExcelWriter(
            OUTPUT_ROOT / "All_FIML_Benchmark.xlsx",
            engine="openpyxl",
        ) as writer:
            if not all_estimates.empty:
                all_estimates.to_excel(
                    writer,
                    sheet_name="All_Estimates",
                    index=False,
                )
            if not all_summaries.empty:
                all_summaries.to_excel(
                    writer,
                    sheet_name="All_Summaries",
                    index=False,
                )
            if not all_comparisons.empty:
                all_comparisons.to_excel(
                    writer,
                    sheet_name="Vs_TwoStage",
                    index=False,
                )
            if not all_agreements.empty:
                all_agreements.to_excel(
                    writer,
                    sheet_name="OptimizerAgreement",
                    index=False,
                )
            selection_df.to_excel(
                writer,
                sheet_name="Selected_2",
                index=False,
            )

    return (
        all_estimates,
        all_summaries,
        all_comparisons,
        all_agreements,
    )


In [10]:

# ============================================================
# APHC: load saved simulation batches and evaluate the direct
# adaptive-progressive-hybrid-censoring Weibull likelihood
# ============================================================

def load_saved_batch_aphc(
    scenario_dir,
    n_pairs,
    n_sim,
    rep_start,
    rep_end,
):
    observed_times_list = []
    actual_removals_list = []
    temperatures = []
    voltages = []

    common_R = None
    common_n = None
    common_t = None
    m = None

    batch_size = rep_end - rep_start

    for pair in range(1, n_pairs + 1):
        path = (
            scenario_dir
            / "Simulated_data"
            / f"pair_{pair}.npz"
        )

        if not path.exists():
            raise FileNotFoundError(path)

        with np.load(path, allow_pickle=False) as z:
            required = {
                "observed_times",
                "actual_removals",
                "n",
                "R",
                "t",
                "temperature",
                "voltage",
            }

            missing = required - set(z.files)
            if missing:
                raise ValueError(
                    f"Missing keys in {path}: {sorted(missing)}"
                )

            obs = np.asarray(
                z["observed_times"],
                dtype=float,
            )[rep_start:rep_end]

            removals = np.asarray(
                z["actual_removals"],
                dtype=int,
            )[rep_start:rep_end]

            R = np.asarray(z["R"], dtype=int)
            n = int(np.asarray(z["n"]).item())
            t = float(np.asarray(z["t"]).item())
            temperature = float(
                np.asarray(z["temperature"]).item()
            )
            voltage = float(
                np.asarray(z["voltage"]).item()
            )

        if common_R is None:
            common_R = R
            common_n = n
            common_t = t
            m = len(R)

            if (
                R.ndim != 1
                or np.any(R < 0)
                or m + int(R.sum()) != n
            ):
                raise ValueError(
                    f"Invalid censoring scheme in {path}"
                )
        else:
            if (
                n != common_n
                or not np.array_equal(R, common_R)
                or not np.isclose(t, common_t)
            ):
                raise ValueError(
                    f"Inconsistent APHC metadata in {path}"
                )

        expected_shape = (
            batch_size,
            m,
        )

        if obs.shape != expected_shape:
            raise ValueError(
                f"Unexpected observed_times shape in {path}: "
                f"{obs.shape}"
            )

        if removals.shape != expected_shape:
            raise ValueError(
                f"Unexpected actual_removals shape in {path}: "
                f"{removals.shape}"
            )

        if (
            np.any(~np.isfinite(obs))
            or np.any(obs <= 0)
            or np.any(np.diff(obs, axis=1) <= 0)
        ):
            raise ValueError(
                f"Invalid/nonordered APHC failures in {path}"
            )

        if (
            np.any(removals < 0)
            or np.any(
                removals.sum(axis=1)
                != n - m
            )
        ):
            raise ValueError(
                f"Invalid APHC actual removals in {path}"
            )

        # Verify that saved actual_removals exactly implement
        # the APHC rule used in the original simulation.
        for r_idx in range(batch_size):
            j = int(np.sum(obs[r_idx] < t))

            if j == m:
                expected = R.copy()
            else:
                expected = np.zeros(m, dtype=int)
                expected[:j] = R[:j]
                expected[-1] = (
                    n - m - int(np.sum(R[:j]))
                )

            if not np.array_equal(
                removals[r_idx],
                expected,
            ):
                raise ValueError(
                    f"APHC removal rule mismatch in {path}, "
                    f"batch row {r_idx}"
                )

        if (
            not np.isfinite(temperature)
            or not np.isfinite(voltage)
            or temperature <= 0
            or voltage <= 0
        ):
            raise ValueError(
                f"Invalid stress metadata in {path}"
            )

        observed_times_list.append(obs)
        actual_removals_list.append(removals)
        temperatures.append(temperature)
        voltages.append(voltage)

    return {
        "observed_times": np.stack(
            observed_times_list, axis=0
        ),
        "actual_removals": np.stack(
            actual_removals_list, axis=0
        ),
        "R": common_R,
        "n": common_n,
        "t": common_t,
        "m": m,
        "temperatures": np.asarray(temperatures, float),
        "voltages": np.asarray(voltages, float),
    }


def build_rep_data_aphc(store, local_rep_index):
    x = store["observed_times"][
        :, local_rep_index, :
    ]

    removals = store["actual_removals"][
        :, local_rep_index, :
    ].astype(float)

    logx = np.log(x)

    return {
        "m": float(store["m"]),
        "logx": logx,
        "sumlogx": logx.sum(axis=1),
        "weights": 1.0 + removals,
    }


def neg_loglik_aphc(
    theta,
    rep_data,
    temperatures,
    voltages,
):
    """
    Negative APHC log-likelihood kernel summed over all stress pairs.

    For stress pair i, all m failures are observed and R*_ik is the
    saved actual removal count:
      l_i = m log(alpha_i)
            - m alpha_i log(lambda_i)
            + (alpha_i-1) sum log(x_ik)
            - sum (1+R*_ik)(x_ik/lambda_i)^alpha_i

    Coefficient-independent progressive-censoring constants are omitted.
    """
    alpha, lam = theta_to_stress_parameters(
        theta,
        temperatures,
        voltages,
    )

    if (
        np.any(~np.isfinite(alpha))
        or np.any(~np.isfinite(lam))
        or np.any(alpha <= 0)
        or np.any(lam <= 0)
    ):
        return PENALTY

    log_lam = np.log(lam)
    m = rep_data["m"]

    h = (
        alpha[:, None]
        * (
            rep_data["logx"]
            - log_lam[:, None]
        )
    )

    if np.any(h > 700):
        return PENALTY

    weighted_power = np.sum(
        rep_data["weights"]
        * np.exp(np.clip(h, -745, 700)),
        axis=1,
    )

    ll_pair = (
        m * np.log(alpha)
        - alpha * m * log_lam
        + (alpha - 1.0)
        * rep_data["sumlogx"]
        - weighted_power
    )

    if not np.isfinite(ll_pair).all():
        return PENALTY

    return float(-ll_pair.sum())


## Run the selected repetition batch

Change only `START_REP` and `END_REP` in the configuration cell for the next batch. Do not delete prior **independent-run** output between batches.

Because the output root is new (`..._INDEPENDENT`), old runs that used two-stage starting values are not reused.

In [11]:

# ============================================================
# Run INDEPENDENT direct-MLE benchmark
# ============================================================

for scenario_number, settings_path in enumerate(
    scenario_paths, start=1
):
    scenario_dir = settings_path.parent
    parameter_name = scenario_dir.parent.name
    dataset_name = scenario_dir.name

    settings_df = pd.read_csv(settings_path)

    if len(settings_df) != 1:
        raise ValueError(
            f"Expected one settings row in {settings_path}"
        )

    settings = settings_df.iloc[0]

    n_pairs = int(settings["N_pairs"])
    n_sim = int(settings["N_simulations"])

    rep_start = max(0, int(START_REP))
    rep_end = min(int(END_REP), n_sim)

    if rep_start >= rep_end:
        raise ValueError(
            f"Invalid repetition batch {rep_start}:{rep_end}"
        )

    # True coefficients are used ONLY after estimation for
    # Monte Carlo error, bias, SD and RMSE calculations.
    true_theta = np.array([
        float(settings["a0"]),
        float(settings["a1"]),
        float(settings["a2"]),
        float(settings["c0"]),
        float(settings["c1"]),
        float(settings["c2"]),
    ])

    print(
        f"\nScenario {scenario_number}/{len(scenario_paths)}: "
        f"{parameter_name} | {dataset_name} | "
        f"repetitions {rep_start+1}-{rep_end}",
        flush=True,
    )

    store = load_saved_batch_aphc(
        scenario_dir,
        n_pairs,
        n_sim,
        rep_start,
        rep_end,
    )

    temperatures = store["temperatures"]
    voltages = store["voltages"]

    # Rank check is a design diagnostic only.
    X = np.column_stack([
        np.ones(n_pairs),
        1.0 / temperatures,
        np.log(voltages),
    ])

    if np.linalg.matrix_rank(X) != 3:
        raise ValueError(
            "Stress predictor matrix is not full rank."
        )

    # Fixed starts: independent of two-stage estimates, pairwise MLEs,
    # true coefficients, and Monte Carlo outcomes.
    independent_starts = independent_start_vectors(
        temperatures,
        voltages,
    )

    output_dir = (
        OUTPUT_ROOT
        / parameter_name
        / dataset_name
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    estimate_file = (
        output_dir
        / "Direct_MLE_by_repetition.csv"
    )

    if estimate_file.exists():
        estimates_df = pd.read_csv(estimate_file)
    else:
        estimates_df = pd.DataFrame()

    completed = set()

    if not estimates_df.empty:
        completed = set(zip(
            estimates_df["Repetition"].astype(int),
            estimates_df["Method"].astype(str),
        ))

    new_rows = []

    for global_idx in range(rep_start, rep_end):
        repetition = global_idx + 1
        local_idx = global_idx - rep_start

        rep_data = build_rep_data_aphc(
            store,
            local_idx,
        )

        def objective_theta(theta):
            return neg_loglik_aphc(
                theta,
                rep_data,
                temperatures,
                voltages,
            )

        for method in METHODS:
            if (repetition, method) in completed:
                continue

            row = {
                "Parameter_setting": parameter_name,
                "Dataset": dataset_name,
                "Censoring": CENSORING,
                "Repetition": repetition,
                "Method": method,
                "Start_source": "independent_fixed_multistart",
            }

            for k, name in enumerate(COEFFICIENTS):
                row[f"{name}_true"] = true_theta[k]

            try:
                fit = fit_direct_mle_multistart(
                    objective_theta,
                    independent_starts,
                    temperatures,
                    voltages,
                    method,
                )

                theta_hat = fit.pop("theta_hat")
                selected_start_theta = fit.pop(
                    "selected_start_theta"
                )
                fit.pop("eta_hat", None)
                fit.pop("standardization", None)

                row.update(fit)

                # Record which independent start produced the retained optimum.
                for k, name in enumerate(COEFFICIENTS):
                    row[f"{name}_start"] = selected_start_theta[k]
                    row[f"{name}_est"] = theta_hat[k]
                    row[f"{name}_error"] = (
                        theta_hat[k] - true_theta[k]
                    )

                if (
                    COMPUTE_CI
                    and row["Fit_success"]
                ):
                    try:
                        row.update(
                            ci_from_hessian(
                                objective_theta,
                                theta_hat,
                                temperatures,
                                voltages,
                            )
                        )
                    except Exception as error:
                        row["CI_success"] = False
                        row["CI_message"] = str(error)
                else:
                    row["CI_success"] = False

            except Exception as error:
                row.update({
                    "Selected_start_id": "",
                    "N_starts_attempted": len(independent_starts),
                    "N_starts_successful": 0,
                    "Fit_success": False,
                    "Optimizer_success": False,
                    "Objective_start": np.nan,
                    "Objective": np.nan,
                    "Objective_improvement": np.nan,
                    "LogLik_kernel": np.nan,
                    "Min_alpha_hat": np.nan,
                    "Min_lambda_hat": np.nan,
                    "Selected_start_runtime_sec": np.nan,
                    "Selected_start_NFEV": np.nan,
                    "Selected_start_NIT": np.nan,
                    "NFEV": np.nan,
                    "NIT": np.nan,
                    "Runtime_sec": np.nan,
                    "Message": str(error),
                    "CI_success": False,
                })

                for name in COEFFICIENTS:
                    row[f"{name}_start"] = np.nan
                    row[f"{name}_est"] = np.nan
                    row[f"{name}_error"] = np.nan

            new_rows.append(row)

        if (
            (global_idx - rep_start + 1)
            % CHECKPOINT_EVERY == 0
            or global_idx == rep_end - 1
        ):
            if new_rows:
                estimates_df = pd.concat(
                    [
                        estimates_df,
                        pd.DataFrame(new_rows),
                    ],
                    ignore_index=True,
                )

                estimates_df = (
                    estimates_df
                    .drop_duplicates(
                        subset=[
                            "Repetition",
                            "Method",
                        ],
                        keep="last",
                    )
                    .sort_values(
                        ["Repetition", "Method"]
                    )
                    .reset_index(drop=True)
                )

                estimates_df.to_csv(
                    estimate_file,
                    index=False,
                )

                new_rows = []

            print(
                f"  checkpoint: repetition "
                f"{repetition}/{n_sim}",
                flush=True,
            )

    if new_rows:
        estimates_df = pd.concat(
            [
                estimates_df,
                pd.DataFrame(new_rows),
            ],
            ignore_index=True,
        )

        estimates_df = (
            estimates_df
            .drop_duplicates(
                subset=[
                    "Repetition",
                    "Method",
                ],
                keep="last",
            )
            .sort_values(
                ["Repetition", "Method"]
            )
            .reset_index(drop=True)
        )

        estimates_df.to_csv(
            estimate_file,
            index=False,
        )

    summary_df = summarize_estimates(
        estimates_df,
        true_theta,
    )

    settings_out = {
        "Censoring": CENSORING,
        "Parameter_setting": parameter_name,
        "Dataset": dataset_name,
        "n": int(settings["n"]),
        "m": int(settings["m"]),
        "Planned_observed_fraction": (
            float(settings["m"])
            / float(settings["n"])
        ),
        "N_pairs": n_pairs,
        "N_simulations": n_sim,
        "Methods": ", ".join(METHODS),
        "Initialization": "independent fixed multi-start",
        "N_independent_starts": len(independent_starts),
        "Independent_start_ids": ", ".join(
            start_id for start_id, _ in independent_starts
        ),
        "Compute_CI": COMPUTE_CI,
        "Source_scenario": str(scenario_dir),
    }

    for k, name in enumerate(COEFFICIENTS):
        settings_out[name] = true_theta[k]

    save_scenario_outputs(
        estimates_df,
        summary_df,
        settings_out,
        output_dir,
        scenario_dir,
        temperatures,
        voltages,
    )

    print(
        "  valid fits:",
        {
            method: int(
                estimates_df.loc[
                    estimates_df["Method"] == method,
                    "Fit_success",
                ]
                .fillna(False)
                .astype(bool)
                .sum()
            )
            for method in METHODS
        },
        flush=True,
    )

    del store
    gc.collect()

    rebuild_global_outputs()

print("\nCurrent independent multi-start batch completed.", flush=True)



Scenario 1/8: baseline | m60_n240_balanced | repetitions 1-100
  checkpoint: repetition 5/1000
  checkpoint: repetition 10/1000
  checkpoint: repetition 15/1000
  checkpoint: repetition 20/1000
  checkpoint: repetition 25/1000
  checkpoint: repetition 30/1000
  checkpoint: repetition 35/1000
  checkpoint: repetition 40/1000
  checkpoint: repetition 45/1000
  checkpoint: repetition 50/1000
  checkpoint: repetition 55/1000
  checkpoint: repetition 60/1000
  checkpoint: repetition 65/1000
  checkpoint: repetition 70/1000
  checkpoint: repetition 75/1000
  checkpoint: repetition 80/1000
  checkpoint: repetition 85/1000
  checkpoint: repetition 90/1000
  checkpoint: repetition 95/1000
  checkpoint: repetition 100/1000
  valid fits: {'Nelder-Mead': 100, 'Powell': 100}

Scenario 2/8: baseline | m180_n240_balanced | repetitions 1-100
  checkpoint: repetition 5/1000
  checkpoint: repetition 10/1000
  checkpoint: repetition 15/1000
  checkpoint: repetition 20/1000
  checkpoint: repetition 25/10

In [12]:

# ============================================================
# Validation / current combined results
# ============================================================

(
    all_estimates,
    all_summaries,
    all_comparisons,
    all_agreements,
) = rebuild_global_outputs()

print("Combined estimate rows:", len(all_estimates))
print("Combined summary rows:", len(all_summaries))

if not all_summaries.empty:
    display(
        all_summaries[
            [
                "Parameter_setting",
                "Dataset",
                "Method",
                "Coefficient",
                "N_repetitions",
                "N_valid_fits",
                "Bias",
                "SD",
                "RMSE",
                "Fit_failure_rate",
                "Mean_runtime_sec",
            ]
        ].head(30)
    )

print(
    "\nFor a full 1,000-repetition run, each completed scenario "
    "should ultimately contain 2,000 rows: "
    "1,000 Nelder-Mead + 1,000 Powell."
)

print("Optimizer agreement rows:", len(all_agreements))


Combined estimate rows: 1600
Combined summary rows: 96


,Parameter_setting,Dataset,Method,Coefficient,N_repetitions,N_valid_fits,Bias,SD,RMSE,Fit_failure_rate,Mean_runtime_sec
0,baseline,m60_n240_balanced,Nelder-Mead,a0,100,100,-0.075921,0.413104,0.420022,0.0,0.228558
1,baseline,m60_n240_balanced,Nelder-Mead,a1,100,100,17.208889,87.806610,89.477073,0.0,0.228558
2,baseline,m60_n240_balanced,Nelder-Mead,a2,100,100,0.006583,0.135336,0.135496,0.0,0.228558
3,baseline,m60_n240_balanced,Nelder-Mead,c0,100,100,0.040872,0.408912,0.410949,0.0,0.228558
4,baseline,m60_n240_balanced,Nelder-Mead,c1,100,100,-21.129303,83.033264,85.679463,0.0,0.228558
5,baseline,m60_n240_balanced,Nelder-Mead,c2,100,100,0.012392,0.144432,0.144963,0.0,0.228558
6,baseline,m60_n240_balanced,Powell,a0,100,100,-0.076314,0.412871,0.419865,0.0,0.241863
7,baseline,m60_n240_balanced,Powell,a1,100,100,17.417545,87.593199,89.308115,0.0,0.241863
8,baseline,m60_n240_balanced,Powell,a2,100,100,0.006464,0.135304,0.135458,0.0,0.241863
9,baseline,m60_n240_balanced,Powell,c0,100,100,0.041363,0.408761,0.410849,0.0,0.241863



For a full 1,000-repetition run, each completed scenario should ultimately contain 2,000 rows: 1,000 Nelder-Mead + 1,000 Powell.
Optimizer agreement rows: 800
